# Live GEE Pipeline — Recompute the Numbers Yourself

This notebook calls Google Earth Engine directly to recompute Mask A,
Mask B and the EUDR-risk conversion mask for a chosen AOI. You need:

- A GEE account with a registered project name.
- `earthengine-api` and `geemap` installed (uncomment the pip line below).
- Internet access.

If you do not have GEE access, the published snapshot in
`data/runs/2026-06-08/` is sufficient to reproduce every number cited
in the companion article. Use that path via
`notebooks/00_reproduce_article_numbers.ipynb` instead.

In [1]:
# %pip install -q earthengine-api geemap

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import ee
# Replace 'your-gee-project' with your registered Cloud project ID.
ee.Initialize(project='your-gee-project')

from src.aois import ee_geometry, get_aoi
from src.gee import conversion_mask, recent_loss_mask, plantation_probability_image

## Pick an AOI

In [ ]:
aoi_id = 'civ_soubre_33km'  # or any aoi_id from src.aois.AOIS
# civ_soubre_33km is not in the default registry — example values shown.
# Replace with an existing AOI like 'cocoa_civ_sw' or 'cocoa_gha_west'.
geom = ee_geometry('cocoa_civ_sw')
print(get_aoi('cocoa_civ_sw'))

## Compute the masks

In [ ]:
mask_a = recent_loss_mask(geom)
prob = plantation_probability_image(geom)
risk = conversion_mask(geom)

# Pixel area in hectares: Hansen native is 30 m, but at the equator a 30 m
# square = 0.09 ha. For accurate area we use ee.Image.pixelArea() in m^2.
pixel_area_ha = ee.Image.pixelArea().divide(10000)

plant_ha = (prob.gte(0.5).multiply(pixel_area_ha)
            .reduceRegion(reducer=ee.Reducer.sum(), geometry=geom, scale=30, maxPixels=1e10)
            .getInfo())
risk_ha = (risk.multiply(pixel_area_ha)
           .reduceRegion(reducer=ee.Reducer.sum(), geometry=geom, scale=30, maxPixels=1e10)
           .getInfo())
print(f'Plantation area (FDP prob >= 0.5): {plant_ha}')
print(f'EUDR-risk area: {risk_ha}')

## Reproduction caveats

- The result will differ slightly between runs as GEE updates the
  underlying assets. The article's quoted numbers (Soubré 2.74 %,
  Sefwi-Wiawso 6.05 %) come from the 2026-06-08 run; later runs may
  shift by a fraction of a percentage point.
- The FDP cocoa layer is **CC-BY-4.0-NC**. Verify your licensing
  position before using derived numbers commercially.
- The pixel-area computation above is approximate at AOI scale. For
  audit-grade area accounting, follow Olofsson et al. 2014 *RSE* good
  practice (stratified random sample + Cohen's kappa).